# 02 — Disease Classification with EfficientNet-B0

Transfer learning classifier on the PlantVillage Kaggle dataset.

**Approach**:
- Architecture: **EfficientNet-B0** pre-trained on ImageNet (via `timm`)
- Data: 80/20 stratified train/test split from `ImageFolder`
- Phase 1: Freeze backbone → train only the classifier head (fast convergence)
- Phase 2: Unfreeze all layers → fine-tune with 10× smaller learning rate
- Mixed-precision (AMP) when CUDA is available
- Evaluation: accuracy, per-class F1, confusion matrix, **Grad-CAM**

Expected accuracy: **>95%** (literature with GoogLeNet: 99.35%)

In [ ]:
# Install dependencies and download dataset
!pip install -q datasets torch torchvision timm opencv-python matplotlib seaborn tqdm
!pip install -q kaggle
!kaggle datasets download -d abdallahalidev/plantvillage-dataset --unzip -p /content/plantvillage

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms, datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter
import cv2

plt.rcParams['figure.dpi'] = 110
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
print(f'Device: {DEVICE}  |  AMP: {USE_AMP}')

## Helper functions (model, transforms, visualisation, Grad-CAM)

In [ ]:
# ── ImageNet normalisation constants ────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def get_transforms(split, image_size=224):
    """Return torchvision transforms for 'train' or 'test' split."""
    if split == 'train':
        return transforms.Compose([
            transforms.Resize((image_size + 32, image_size + 32)),
            transforms.RandomCrop(image_size),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.3, contrast=0.3,
                                   saturation=0.3, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


# ── Model factory ────────────────────────────────────────────────────────────
def build_model(num_classes, model_name='efficientnet_b0',
                pretrained=True, drop_rate=0.3):
    import timm
    return timm.create_model(model_name, pretrained=pretrained,
                             num_classes=num_classes, drop_rate=drop_rate)


def freeze_backbone(model):
    for name, param in model.named_parameters():
        if 'classifier' not in name:
            param.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'[model] Backbone frozen — trainable params: {trainable:,}')


def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'[model] All layers unfrozen — trainable params: {trainable:,}')


def get_gradcam_target_layer(model, model_name='efficientnet_b0'):
    if 'efficientnet' in model_name:
        return model.blocks[-1]
    if 'resnet' in model_name:
        return model.layer4[-1]
    last_conv = None
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            last_conv = m
    return last_conv


# ── Tensor helpers ───────────────────────────────────────────────────────────
_MEAN = np.array(IMAGENET_MEAN)
_STD  = np.array(IMAGENET_STD)

def denormalize(tensor):
    """ImageNet-normalised tensor [3,H,W] → uint8 numpy [H,W,3]."""
    img = tensor.permute(1, 2, 0).cpu().numpy()
    img = img * _STD + _MEAN
    return (np.clip(img, 0, 1) * 255).astype(np.uint8)


# ── Training curves ──────────────────────────────────────────────────────────
def plot_training_curves(history):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, history['train_loss'], 'b-o', label='Train loss',  markersize=4)
    ax1.plot(epochs, history['val_loss'],   'r-o', label='Val loss',    markersize=4)
    ax1.set_title('Loss'); ax1.set_xlabel('Epoch')
    ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(epochs, history['train_acc'], 'b-o', label='Train acc',  markersize=4)
    ax2.plot(epochs, history['val_acc'],   'r-o', label='Val acc',    markersize=4)
    ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch')
    ax2.legend(); ax2.grid(alpha=0.3)

    plt.suptitle('Training Curves — EfficientNet-B0 on PlantVillage', fontsize=12)
    plt.tight_layout()
    return fig


# ── Grad-CAM ─────────────────────────────────────────────────────────────────
def compute_gradcam(model, image_tensor, target_layer,
                    class_idx=None, device='cpu'):
    model.eval()
    inp = image_tensor.unsqueeze(0).to(device)
    gradients, activations = [], []

    def _fwd_hook(_module, _inp, output):
        activations.append(output)
        output.register_hook(gradients.append)

    handle = target_layer.register_forward_hook(_fwd_hook)
    output = model(inp)
    if class_idx is None:
        class_idx = int(output.argmax(dim=1).item())
    model.zero_grad()
    output[0, class_idx].backward()
    handle.remove()

    grads   = gradients[0].cpu().numpy()[0]
    acts    = activations[0].detach().cpu().numpy()[0]
    weights = grads.mean(axis=(1, 2))
    heatmap = np.sum(weights[:, None, None] * acts, axis=0)
    heatmap = np.maximum(heatmap, 0)
    heatmap /= (heatmap.max() + 1e-8)
    return heatmap.astype(np.float32)


def apply_gradcam(original_image, heatmap, alpha=0.50):
    h, w    = original_image.shape[:2]
    resized = cv2.resize(heatmap, (w, h))
    jet     = cv2.applyColorMap((resized * 255).astype(np.uint8), cv2.COLORMAP_JET)
    jet_rgb = cv2.cvtColor(jet, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(original_image, 1 - alpha, jet_rgb, alpha, 0)


def plot_gradcam_grid(tensors, heatmaps, predictions, ground_truths, figsize=(20, 5)):
    n = len(tensors)
    fig, axes = plt.subplots(2, n, figsize=figsize)
    for i in range(n):
        orig = denormalize(tensors[i])
        axes[0][i].imshow(orig);                  axes[0][i].axis('off')
        axes[0][i].set_title(f'GT: {ground_truths[i]}', fontsize=7)
        overlay = apply_gradcam(orig, heatmaps[i])
        axes[1][i].imshow(overlay);               axes[1][i].axis('off')
        axes[1][i].set_title(f'Pred: {predictions[i]}', fontsize=7)
    axes[0][0].set_ylabel('Original', fontsize=9, rotation=90)
    axes[1][0].set_ylabel('Grad-CAM', fontsize=9, rotation=90)
    plt.suptitle('Grad-CAM Visualisation', fontsize=12)
    plt.tight_layout()
    return fig

print('Helper functions defined.')

## 1. Load dataset and create 80/20 stratified split

In [ ]:
DATA_COLOR = '/content/plantvillage/color'

# Load full dataset with test transform first (we'll apply train transform to subset later)
full_dataset = datasets.ImageFolder(DATA_COLOR)
class_names  = full_dataset.classes
num_classes  = len(class_names)
targets      = full_dataset.targets

# Stratified 80/20 split on indices
train_idx, test_idx = train_test_split(
    range(len(full_dataset)),
    test_size=0.2,
    stratify=targets,
    random_state=42,
)

# Create two ImageFolder datasets with different transforms
train_dataset = datasets.ImageFolder(DATA_COLOR, transform=get_transforms('train'))
test_dataset  = datasets.ImageFolder(DATA_COLOR, transform=get_transforms('test'))

train_ds = Subset(train_dataset, train_idx)
test_ds  = Subset(test_dataset,  test_idx)

BATCH_SIZE  = 32
NUM_WORKERS = 2  # Colab runs Linux — multiprocessing supported

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'))
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'))

print(f'Total images    : {len(full_dataset):,}')
print(f'Train samples   : {len(train_ds):,}')
print(f'Test  samples   : {len(test_ds):,}')
print(f'Classes         : {num_classes}')
print(f'Train batches   : {len(train_loader)}')
print(f'Test  batches   : {len(test_loader)}')

## 2. Build model

In [ ]:
MODEL_NAME = 'efficientnet_b0'
model = build_model(num_classes=num_classes, model_name=MODEL_NAME).to(DEVICE)

total_p = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_p:,}')
print('\nClassifier head:')
print(model.classifier)

## 3. Training

> **Note**: Training on CPU takes several hours.  
> On a Colab T4 GPU it takes ~25–35 min for 15 epochs total.  
> If you only want evaluation, skip to **Section 4** and load a checkpoint.

In [ ]:
PHASE1_EPOCHS = 5
PHASE2_EPOCHS = 10
LR            = 1e-3
criterion     = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler        = torch.cuda.amp.GradScaler(enabled=USE_AMP)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0


def run_epoch(model, loader, criterion, optimizer=None, train=True):
    model.train() if train else model.eval()
    total_loss = correct = total = 0
    all_preds, all_labels = [], []

    ctx = torch.enable_grad if train else torch.no_grad
    with ctx():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if train:
                optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                out  = model(imgs)
                loss = criterion(out, labels)
            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            preds = out.argmax(1)
            total_loss += loss.item() * imgs.size(0)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / total, correct / total, all_preds, all_labels


# ── Phase 1: warm-up (backbone frozen) ──────────────────────────────────────
print('=== Phase 1: head warm-up (backbone frozen) ===')
freeze_backbone(model)
opt1 = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
sch1 = CosineAnnealingLR(opt1, T_max=PHASE1_EPOCHS)

for epoch in range(1, PHASE1_EPOCHS + 1):
    tl, ta, _, _      = run_epoch(model, train_loader, criterion, opt1, train=True)
    vl, va, vp, vlbls = run_epoch(model, test_loader,  criterion, train=False)
    sch1.step()
    history['train_loss'].append(tl); history['train_acc'].append(ta)
    history['val_loss'].append(vl);   history['val_acc'].append(va)
    print(f'[P1 {epoch}/{PHASE1_EPOCHS}] loss {tl:.4f}/{vl:.4f}  acc {ta:.4f}/{va:.4f}')
    if va > best_val_acc:
        best_val_acc = va
        best_preds, best_labels = vp, vlbls
        os.makedirs('/content/runs/notebook', exist_ok=True)
        torch.save(model.state_dict(), '/content/runs/notebook/best_model.pth')

# ── Phase 2: full fine-tuning ────────────────────────────────────────────────
print()
print('=== Phase 2: full fine-tuning ===')
unfreeze_all(model)
opt2 = AdamW(model.parameters(), lr=LR / 10)
sch2 = CosineAnnealingLR(opt2, T_max=PHASE2_EPOCHS)

for epoch in range(1, PHASE2_EPOCHS + 1):
    tl, ta, _, _      = run_epoch(model, train_loader, criterion, opt2, train=True)
    vl, va, vp, vlbls = run_epoch(model, test_loader,  criterion, train=False)
    sch2.step()
    history['train_loss'].append(tl); history['train_acc'].append(ta)
    history['val_loss'].append(vl);   history['val_acc'].append(va)
    print(f'[P2 {epoch}/{PHASE2_EPOCHS}] loss {tl:.4f}/{vl:.4f}  acc {ta:.4f}/{va:.4f}')
    if va > best_val_acc:
        best_val_acc = va
        best_preds, best_labels = vp, vlbls
        torch.save(model.state_dict(), '/content/runs/notebook/best_model.pth')
        print(f'  ✓  Best: {best_val_acc:.4f}')

with open('/content/runs/notebook/history.json', 'w') as f:
    json.dump(history, f)
with open('/content/runs/notebook/class_names.json', 'w') as f:
    json.dump(class_names, f)

print(f'\nBest validation accuracy: {best_val_acc:.4f}')

## 4. Load checkpoint (if training was skipped)

In [ ]:
# Run this cell only if you skipped the training cell above.
RUN_DIR = '/content/runs/notebook'   # change if you uploaded a checkpoint

if 'best_preds' not in dir():
    with open(os.path.join(RUN_DIR, 'class_names.json')) as f:
        class_names = json.load(f)
    with open(os.path.join(RUN_DIR, 'history.json')) as f:
        history = json.load(f)

    model = build_model(num_classes=len(class_names), model_name=MODEL_NAME).to(DEVICE)
    model.load_state_dict(
        torch.load(os.path.join(RUN_DIR, 'best_model.pth'), map_location=DEVICE)
    )
    model.eval()

    criterion = nn.CrossEntropyLoss()
    _, _, best_preds, best_labels = run_epoch(model, test_loader, criterion, train=False)
    print('Checkpoint loaded.')
else:
    print('Using model trained in this session.')

## 5. Training curves

In [ ]:
fig = plot_training_curves(history)
plt.savefig('training_curves.png', bbox_inches='tight')
plt.show()

## 6. Classification report

In [ ]:
print(classification_report(best_labels, best_preds,
                            target_names=class_names, digits=3))

## 7. Confusion matrix

In [ ]:
cm = confusion_matrix(best_labels, best_preds)

fig, ax = plt.subplots(figsize=(18, 16))
sns.heatmap(
    cm, annot=False, fmt='d',
    xticklabels=class_names, yticklabels=class_names,
    cmap='Blues', ax=ax, linewidths=0.3,
)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True',      fontsize=11)
ax.set_title('Confusion Matrix — PlantVillage Test Set', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(rotation=0,  fontsize=7)
plt.tight_layout()
plt.savefig('confusion_matrix.png', bbox_inches='tight')
plt.show()

## 8. Top misclassifications

In [ ]:
errors = [
    (class_names[t], class_names[p])
    for t, p in zip(best_labels, best_preds) if t != p
]
top_errors = Counter(errors).most_common(10)

print(f'Total test errors: {len(errors)} / {len(best_preds)} '
      f'({100*len(errors)/len(best_preds):.1f}%)')
print('\nTop-10 confusion pairs (True → Predicted):')
for (true_c, pred_c), cnt in top_errors:
    print(f'  {cnt:3d}×  {true_c:<40} → {pred_c}')

## 9. Grad-CAM visualisation

Grad-CAM shows **which regions** of the leaf the model uses to make its prediction.  
Hot colours (red/yellow) indicate high importance.

In [ ]:
model.eval()
target_layer = get_gradcam_target_layer(model, MODEL_NAME)
print(f'Grad-CAM target layer: {type(target_layer).__name__}')

N_VIS = 8
batch_tensors, batch_labels = next(iter(test_loader))
selected = batch_tensors[:N_VIS]
sel_lbls = batch_labels[:N_VIS]

tensors_vis, heatmaps_vis, predictions_vis, gt_vis = [], [], [], []

for i in range(N_VIS):
    tensor  = selected[i]
    heatmap = compute_gradcam(model, tensor, target_layer,
                              class_idx=None, device=str(DEVICE))
    with torch.no_grad():
        logits   = model(tensor.unsqueeze(0).to(DEVICE))
        pred_idx = logits.argmax(1).item()

    tensors_vis.append(tensor)
    heatmaps_vis.append(heatmap)

    pred_lbl = class_names[pred_idx]
    gt_lbl   = class_names[sel_lbls[i].item()]
    predictions_vis.append(pred_lbl.split('___')[1] if '___' in pred_lbl else pred_lbl)
    gt_vis.append(gt_lbl.split('___')[1]             if '___' in gt_lbl   else gt_lbl)

fig = plot_gradcam_grid(tensors_vis, heatmaps_vis, predictions_vis, gt_vis, figsize=(22, 6))
plt.savefig('gradcam_grid.png', bbox_inches='tight')
plt.show()

In [ ]:
# Single detailed Grad-CAM example
idx     = 0
tensor  = selected[idx]
orig    = denormalize(tensor)
heatmap = compute_gradcam(model, tensor, target_layer, device=str(DEVICE))
overlay = apply_gradcam(orig, heatmap)

with torch.no_grad():
    logits = model(tensor.unsqueeze(0).to(DEVICE))
    probs  = F.softmax(logits, dim=1)[0]
pred_class = class_names[probs.argmax().item()]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(orig);     axes[0].set_title('Original');       axes[0].axis('off')
axes[1].imshow(overlay);  axes[1].set_title(f'Grad-CAM\nPred: {pred_class}'); axes[1].axis('off')
im = axes[2].imshow(heatmap, cmap='jet')
axes[2].set_title('Heatmap (raw)');  axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046)
plt.suptitle(f'Ground truth: {class_names[sel_lbls[idx].item()]}', fontsize=11)
plt.tight_layout()
plt.savefig('gradcam_detail.png', bbox_inches='tight')
plt.show()

---
## Summary

| Metric | Value |
|--------|-------|
| Architecture | EfficientNet-B0 |
| Parameters | ~5.3M |
| Training images | ~43,500 (80%) |
| Test images | ~10,800 (20%) |
| Expected val accuracy | >95% |
| Interpretability | Grad-CAM |

**Next**: `03_leaf_area.ipynb` — leaf segmentation and disease area quantification.